In [4]:
t = "/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train_holdout_0/hsp60_train_holdout_0_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_0/PredictedLabels/Coords_All/24092018.coords"
res_18 = pd.read_csv(t, delimiter="\t", header=None)
res_18.columns = ["label", "x", "y", "z"]

In [6]:
import sys 
sys.path.append("./srcs")

In [21]:
import utils 
import pickle
with open("./release_assets/metadata/new_hsp60/centers3.pkl", "rb") as f:
    hsp60_centers = pickle.load(f)
labels = hsp60_centers["24092018"]

            # res_true = pd.read_csv("/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/raw_data/24092018.txt", sep="\t", header=None)
            # res_true.columns = ["x", "y", "z"]
a, b, c = utils.match_and_find_closest(res_18[["z", "y", "x"]].values, labels)

In [29]:
import pandas as pd 
import numpy as np
work = {
    "24092001":"/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60_new/24092001_.coords", 
    "24092002":"/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60_new/24092002_.coords", 
    "24092018":"/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60_new/24092018_.coords",
    "24092024":"/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60_new/24092024_.coords",
    "24092028":"/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60_new/24092028_.coords", 
    "24092031":"/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60_new/24092031_.coords"
}
res = {}
for i in work.keys():
    predict = pd.read_csv(work[i], header=None, sep="\t")
    predict_center = predict[[3, 2, 1]].values
    matches, match_dists, closest = utils.match_and_find_closest(predict_center, hsp60_centers[i])
    # score = -predict[2].values
    # t = postprocess.calculate_metrics(matches, match_dists, score, threshold=15)
    cnts = np.sum(match_dists < 20)
    recall = cnts / len(hsp60_centers[i]) 
    precision = cnts / len(predict_center)
    f1 = 2*recall*precision / (recall+precision)
    print(i, cnts, recall, precision, f1)
    res[i] = {
        "recall":recall, 
        "precision":precision,
        "f1":f1
    }
# with open("./release_assets/metadata/particle_res/deepET_f1.pkl", "wb") as f:
#     pickle.dump(res, f)

24092001 265 0.8255451713395638 0.4514480408858603 0.5837004405286343
24092002 152 0.5588235294117647 0.38190954773869346 0.45373134328358206
24092018 136 0.5506072874493927 0.28451882845188287 0.3751724137931035
24092024 215 0.6913183279742765 0.3687821612349914 0.4809843400447426
24092028 46 0.5168539325842697 0.2026431718061674 0.2911392405063291
24092031 187 0.6192052980132451 0.3617021276595745 0.45665445665445664


In [30]:
import pickle
import sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.append("./srcs")
import utils

# Ground-truth ribosome centers
with open("/virtual_data/train_data/training/cr/temp/ribo_centers_new.pkl", "rb") as f:
    ribo_centers = pickle.load(f)

# Prediction files from the two new folds (each tomo was held-out exactly once)
base = Path("/virtual_data/test_data/dxd20240920/DeepETPicker/ribosome/runs")

fold1_test_dir = sorted(base.glob("ribosome_fold1_train_fold2_test/**/PredictedLabels/Coords_All"))[0]
fold2_test_dir = sorted(base.glob("ribosome_fold2_train_fold1_test/**/PredictedLabels/Coords_All"))[0]

# Map each tomo_id to its prediction file (from whichever fold held it out)
pred_map = {}
for p in fold1_test_dir.glob("*.coords"):
    pred_map[p.stem] = p
for p in fold2_test_dir.glob("*.coords"):
    pred_map[p.stem] = p

print(f"Prediction files found: {sorted(pred_map.keys())}")
missing = set(ribo_centers.keys()) - set(pred_map.keys())
if missing:
    print(f"WARNING: no prediction for {missing}")

# Match distance threshold (voxels) — same as hsp60 assessment above
MATCH_THRESHOLD = 20

res = {}
print(f"\n{'tomo':>12}  {'#gt':>5}  {'#pred':>5}  {'cnt':>5}  {'precision':>10}  {'recall':>8}  {'f1':>8}")
print("-" * 70)

for tomo_id in sorted(ribo_centers.keys()):
    if tomo_id not in pred_map:
        continue

    predict = pd.read_csv(pred_map[tomo_id], header=None, sep="\t")
    predict_center = predict[[3, 2, 1]].values   # [x, y, z] — same column order as hsp60 assess

    labels = np.array(ribo_centers[tomo_id])

    _, match_dists, _ = utils.match_and_find_closest(predict_center, labels)
    cnt = int(np.sum(match_dists < MATCH_THRESHOLD))

    precision = cnt / len(predict_center) if len(predict_center) > 0 else 0.0
    recall = cnt / len(labels) if len(labels) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    res[tomo_id] = {"precision": precision, "recall": recall, "f1": f1,
                    "gt": len(labels), "pred": len(predict_center), "cnt": cnt}
    print(f"{tomo_id:>12}  {len(labels):>5}  {len(predict_center):>5}  {cnt:>5}  {precision:>10.4f}  {recall:>8.4f}  {f1:>8.4f}")

# Micro-average over all 14 tomograms
tp = sum(v["cnt"] for v in res.values())
total_pred = sum(v["pred"] for v in res.values())
total_gt = sum(v["gt"] for v in res.values())
micro_prec = tp / total_pred if total_pred else 0.0
micro_rec = tp / total_gt if total_gt else 0.0
micro_f1 = 2 * micro_prec * micro_rec / (micro_prec + micro_rec) if (micro_prec + micro_rec) > 0 else 0.0

print("-" * 70)
print(f"{'MICRO':>12}  {total_gt:>5}  {total_pred:>5}  {tp:>5}  {micro_prec:>10.4f}  {micro_rec:>8.4f}  {micro_f1:>8.4f}")
print(f"\nDeepETPicker ribosome 2-fold  micro-F1 = {micro_f1:.6f}")

Prediction files found: ['24092002', '24092009', '24092013', '24092014', '24092016', '24092017', '24092020', '24092023', '24092024', '24092025', '24092026', '24092029', '24092030', '24092031']

        tomo    #gt  #pred    cnt   precision    recall        f1
----------------------------------------------------------------------
    24092002    388   1093    337      0.3083    0.8686    0.4551
    24092009    759   2369    681      0.2875    0.8972    0.4354
    24092013   1453   2420   1176      0.4860    0.8094    0.6073
    24092014   1049   2218    979      0.4414    0.9333    0.5993
    24092016   1298   2807   1205      0.4293    0.9284    0.5871
    24092017   1074   2489    993      0.3990    0.9246    0.5574
    24092020    479   1672    431      0.2578    0.8998    0.4007
    24092023    791   1839    759      0.4127    0.9595    0.5772
    24092024    348   1100    324      0.2945    0.9310    0.4475
    24092025    294   2055    280      0.1363    0.9524    0.2384
    24092